# Lab 02 — Environment Setup and utils.py

**Knowledge base:** `knowledge_base/02_llms/01_how_llms_work.md`

**Concepts:** `.env` file · `python-dotenv` · Ollama · Gemini · `utils.py` functions

This lab explains exactly what `utils.py` does and confirms your LLM connection works
before you use it in every lab that follows.

---

## 1 — Your `.env` file

The `.env` file lives at `D:\AI_Engineer-RAG\.env`.
It tells `utils.py` which backend to use and where to find API keys.

```
# .env — choose ONE backend (uncomment only one LLM_BACKEND line)
LLM_BACKEND=ollama          ← local Ollama, no API key needed
# LLM_BACKEND=gemini        ← Google Gemini (needs GEMINI_API_KEY)

# Ollama settings — make sure `ollama serve` is running
OLLAMA_MODEL=qwen2.5:7b
OLLAMA_HOST=http://localhost:11434

# Gemini settings — get a free key at https://aistudio.google.com/apikey
GEMINI_API_KEY=your_gemini_key_here
GEMINI_MODEL=gemini-2.0-flash
```

**Ollama quick start (if not already done):**
```
ollama pull qwen2.5:7b
ollama serve
```

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)   # reads D:\AI_Engineer-RAG\.env

backend = os.environ.get("LLM_BACKEND", "ollama")
print(f"✅ LLM_BACKEND: {backend}")

if backend == "ollama":
    host  = os.environ.get("OLLAMA_HOST",  "http://localhost:11434")
    model = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
    print(f"   Host:  {host}")
    print(f"   Model: {model}")
    print("   → Make sure `ollama serve` is running in a terminal.")
elif backend == "gemini":
    key = os.environ.get("GEMINI_API_KEY", "")
    print(f"   GEMINI_API_KEY: {key[:8]}..." if key else "   ❌ GEMINI_API_KEY not found in .env")
else:
    print(f"❌ Unknown backend '{backend}'. Set LLM_BACKEND=ollama or LLM_BACKEND=gemini")

✅ LLM_BACKEND: ollama
   Host:  http://localhost:11434
   Model: qwen3.5:4B
   → Make sure `ollama serve` is running in a terminal.


---
## 2 — What utils.py does

`utils.py` (in this labs/ folder) is a thin wrapper that hides all API differences.
You call ONE function regardless of whether you're using Ollama or Gemini.

```
generate_with_single_input(prompt)          → single question, single answer
generate_with_multiple_input(messages)      → full conversation with history
```

Both always return: `{'role': 'assistant', 'content': '<response text>'}`

The backend is controlled by `LLM_BACKEND` in your `.env`. Switching backends
means changing one line in `.env` — no code changes anywhere.

In [2]:
# Import the two functions you will use in every lab
import sys
sys.path.insert(0, ".")   # ensures utils.py in this folder is found

from utils import generate_with_single_input, generate_with_multiple_input

print("✅ utils.py imported successfully")
print(f"   Active backend: {os.environ.get('LLM_BACKEND', 'ollama')}")

✅ utils.py imported successfully
   Active backend: ollama


---
## 3 — generate_with_single_input

Use this when you have one standalone question or instruction.

**Parameters:**

| Parameter | Type | Default | What it controls |
|-----------|------|---------|-----------------|
| `prompt` | str | required | The text to send |
| `role` | str | `'user'` | Who is sending — `'user'` or `'system'` |
| `max_tokens` | int | 500 | Max response length |
| `temperature` | float | None | 0.0 = deterministic · 1.0+ = creative |
| `model` | str | backend default | Override the model name |

**Returns:** `{'role': 'assistant', 'content': '...'}`

In [3]:
# Simplest possible call — one question, one answer
output = generate_with_single_input(
    prompt="What is the capital of Egypt?"
)

# The return is always a dict with 'role' and 'content'
print(f"Role:    {output['role']}")
print(f"Content: {output['content']}")

Role:    assistant
Content: The capital of Egypt is **Cairo**.


In [4]:
# max_tokens controls response length
short = generate_with_single_input(
    prompt="Explain what an LLM is.",
    max_tokens=40
)
print("Short (max_tokens=40):")
print(short['content'])

print()

long = generate_with_single_input(
    prompt="Explain what an LLM is.",
    max_tokens=250
)
print("Long (max_tokens=250):")
print(long['content'])

Short (max_tokens=40):


Long (max_tokens=250):



In [5]:
# temperature controls randomness
# temperature=0.0 → the model gives the same answer every time (good for RAG)
# temperature=1.0 → the model gives different answers each run (creative tasks)

print("=== temperature=0.0 (run this cell twice — same result) ===")
det = generate_with_single_input(prompt="Give me one word: a colour.", temperature=0.0)
print(det['content'])

print()
print("=== temperature=1.0 (run this cell twice — likely different result) ===")
creative = generate_with_single_input(prompt="Give me one word: a colour.", temperature=1.0)
print(creative['content'])

=== temperature=0.0 (run this cell twice — same result) ===
Red

=== temperature=1.0 (run this cell twice — likely different result) ===



---
## 4 — generate_with_multiple_input

Use this when you need:
- A **system message** to shape the model's behaviour
- **Multi-turn chat** where the model needs to remember earlier messages
- **Conversation history** to pass to the model

You pass a list of message dicts. The model sees the whole list at once.

In [6]:
# System message — sets how the model behaves across the entire conversation
messages_with_system = [
    {
        "role": "system",
        "content": "You are a concise assistant. Answer in one sentence only. No extra explanation."
    },
    {
        "role": "user",
        "content": "What is Retrieval Augmented Generation?"
    }
]

output = generate_with_multiple_input(messages=messages_with_system)
print(output['content'])

In [7]:
# Multi-turn conversation — the model uses full history to answer the final question
messages = [
    {"role": "user",      "content": "My name is Abdel and I'm learning about RAG."},
    {"role": "assistant", "content": "Great! RAG combines retrieval and generation. What would you like to know?"},
    {"role": "user",      "content": "What did I just tell you my name was?"},
]

output = generate_with_multiple_input(messages=messages, max_tokens=80)
print(output['content'])

In [8]:
# Reusable chat() function — you will use this pattern in labs 04 and 05
history = [
    {"role": "system", "content": "You are a helpful assistant teaching RAG concepts."}
]

def chat(user_message: str, history: list) -> str:
    """Send a message, append both the message and the response to history, return answer."""
    history.append({"role": "user", "content": user_message})
    response = generate_with_multiple_input(messages=history, max_tokens=120)
    history.append({"role": "assistant", "content": response["content"]})
    return response["content"]


answer1 = chat("What is the retrieval step in RAG?", history)
print("Turn 1:", answer1)

answer2 = chat("And what does the generation step do with what was retrieved?", history)
print("\nTurn 2:", answer2)

Turn 1: In the RAG (Retrieval

Turn 2: 


---
## 5 — Live connection test

Run this cell to confirm your chosen backend is fully working.
If it fails, the error message will tell you exactly what to fix.

In [9]:
print(f"Backend: {os.environ.get('LLM_BACKEND', 'ollama')}")
print("Sending test prompt...")

try:
    test = generate_with_single_input(
        prompt="Reply with exactly: CONNECTION OK",
        max_tokens=20,
        temperature=0.0
    )
    print(f"\n✅ Response received: {test['content']}")
except Exception as e:
    print(f"\n❌ Connection failed: {e}")
    backend = os.environ.get("LLM_BACKEND", "ollama")
    if backend == "ollama":
        print("\nFix: run  `ollama serve`  in a separate terminal, then re-run this cell.")
    elif backend == "gemini":
        print("\nFix: check GEMINI_API_KEY in your .env file.")

Backend: ollama
Sending test prompt...

✅ Response received: 


---
**Lab 02 complete.**

Your LLM connection is working. You understand exactly what the two `utils.py` functions do
and what parameters control the output.

**Next:** `lab03_llm_single_calls.ipynb` — experiment with single-call patterns in depth.